In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

import os
os.listdir(); os.chdir("/home/azm0269@auburn.edu/clover/")

from clover.utils.utils import notebook_line_magic
notebook_line_magic()


# DPOK: diffusion policy optimization with KL regularization

This notebook follows the DDPO baseline structure, but replaces the PPO clipped update with the DPOK policy-gradient objective from Eq. 9 of Fan et al. (2023). DPOK treats the KL to the frozen pre-trained diffusion model as an online regularizer at every denoising transition.


In [ ]:
from __future__ import annotations

import gc
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Callable

import torch
from PIL import Image
from torch import Tensor
from tqdm.auto import trange

from clover.utils.rewards_utils import aesthetic_proxy_reward

from clover.utils.baseline_utils import (
    clear_optimizer_state,
    clone_trainable_parameters,
    decode_latents,
    ddpm_mean_std,
    ddpm_step_with_log_prob,
    encode_prompts,
    finite_trainable_parameters,
    gaussian_kl,
    load_lora_pipeline,
    load_reference_pipeline,
    normalize_advantages,
    predict_noise,
    resolve_gpu_ids,
    restore_trainable_parameters,
    safe_metric_mean,
    sample_prompt_batch,
    save_lora_weights,
    set_seed,
    trainable_parameters,
    unet_config,
)


In [ ]:
@dataclass
class DPOKConfig:
    model_id: str = "runwayml/stable-diffusion-v1-5"
    output_dir: str = "outputs/dpok"
    seed: int = 17
    gpu_ids: list[int] = field(default_factory=lambda: [0, 1, 2])
    use_data_parallel: bool = False

    prompt: str = "a colorful clover field at sunrise, high detail"
    negative_prompt: str = "blurry, low quality, distorted"
    train_prompts: tuple[str, ...] = (
        "a colorful clover field at sunrise, high detail",
        "a close-up photo of a bright green clover leaf with dew",
        "a small robot holding a clover in a clean studio photo",
        "an impressionist painting of clovers under warm sunlight",
    )

    height: int = 512
    width: int = 512
    num_inference_steps: int = 30
    guidance_scale: float = 7.5
    eta: float = 1.0

    rollouts_per_epoch: int = 1
    train_epochs: int = 4
    dpok_epochs: int = 1
    minibatch_size: int = 1
    learning_rate: float = 1e-9
    adam_epsilon: float = 1e-4
    lora_rank: int = 2
    lora_alpha: int = 2
    lora_dropout: float = 0.0
    lora_target_modules: tuple[str, ...] = ("to_v",)
    reward_weight: float = 1.0
    kl_weight: float = 0.01
    normalize_rewards: bool = False
    max_grad_norm: float = 0.1
    mixed_precision: bool = True
    gradient_checkpointing: bool = True

    log_every: int = 1
    save_every: int = 5


cfg = DPOKConfig()
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
cfg


In [ ]:
gpu_ids = resolve_gpu_ids(cfg)
device = torch.device(f"cuda:{gpu_ids[0]}" if gpu_ids else "cpu")
dtype = torch.float16 if device.type == "cuda" and cfg.mixed_precision else torch.float32
generator = set_seed(cfg.seed, device)
print(device, dtype, f"gpu_ids={gpu_ids}")


In [ ]:
reward_fn: \
    Callable[[list[Image.Image], list[str]], Tensor] = \
        lambda images, prompts: aesthetic_proxy_reward(images, prompts, device=device)


In [ ]:
pipe = load_lora_pipeline(cfg, device=device, dtype=dtype, gpu_ids=gpu_ids)
reference_pipe = load_reference_pipeline(cfg, device=device, dtype=dtype)
reference_pipe.scheduler.set_timesteps(cfg.num_inference_steps, device=device)

lora_parameters = trainable_parameters(pipe.unet)
print(f"Training {sum(parameter.numel() for parameter in lora_parameters):,} LoRA parameters")
optimizer = torch.optim.AdamW(lora_parameters, lr=cfg.learning_rate, eps=cfg.adam_epsilon)
vae_scale_factor = 2 ** (len(pipe.vae.config.block_out_channels) - 1)


In [ ]:
@torch.no_grad()
def collect_rollouts(pipe, batch_size: int) -> dict[str, Tensor | list[str] | list[Image.Image]]:
    pipe.unet.eval()
    prompts = sample_prompt_batch(cfg.train_prompts, batch_size)
    prompt_embeds = encode_prompts(pipe, prompts, cfg.negative_prompt, device, dtype)
    pipe.scheduler.set_timesteps(cfg.num_inference_steps, device=device)

    latent_shape = (
        batch_size,
        unet_config(pipe).in_channels,
        cfg.height // vae_scale_factor,
        cfg.width // vae_scale_factor,
    )
    latents = torch.randn(latent_shape, generator=generator, device=device, dtype=dtype)
    latents = latents * pipe.scheduler.init_noise_sigma

    states, actions, log_probs, timesteps, rewards = [], [], [], [], []
    images = []
    zero_rewards = torch.zeros(batch_size, dtype=torch.float32)
    for t in pipe.scheduler.timesteps:
        timestep = int(t.item())
        states.append(latents.detach().float().cpu())
        noise_pred = predict_noise(pipe, latents, t, prompt_embeds, cfg.guidance_scale)
        next_latents, log_prob = ddpm_step_with_log_prob(pipe.scheduler, noise_pred, timestep, latents, generator, eta=cfg.eta)
        actions.append(next_latents.detach().float().cpu())
        log_probs.append(log_prob.detach().float().cpu())
        timesteps.append(timestep)
        latents = next_latents
        if timestep == 0:
            images = decode_latents(pipe, latents)
            rewards.append(reward_fn(images, prompts).detach().float().cpu())
        else:
            rewards.append(zero_rewards.clone())

    if not images:
        images = decode_latents(pipe, latents)
    pipe.unet.train()

    return {
        "prompts": prompts,
        "states": torch.stack(states, dim=1),
        "actions": torch.stack(actions, dim=1),
        "old_log_probs": torch.stack(log_probs, dim=1),
        "timesteps": torch.tensor(timesteps, dtype=torch.long),
        "rewards": torch.stack(rewards, dim=1),
        "images": images,
    }


In [ ]:
def dpok_update(
    pipe,
    reference_pipe,
    rollout: dict[str, Tensor | list[str]],
    optimizer: torch.optim.Optimizer,
) -> dict[str, float]:
    states = rollout["states"]
    actions = rollout["actions"]
    timesteps = rollout["timesteps"].tolist()
    rewards = rollout["rewards"]
    prompts = rollout["prompts"]
    terminal_rewards = rewards.sum(dim=1)
    policy_weights = normalize_advantages(terminal_rewards) if cfg.normalize_rewards else terminal_rewards
    policy_weights = policy_weights.to(device)

    batch_size, trajectory_len = states.shape[:2]
    indices = torch.arange(batch_size)
    losses, policy_losses, kl_losses, kls = [], [], [], []
    skipped_updates = 0

    pipe.unet.train()
    reference_pipe.unet.eval()
    if not finite_trainable_parameters(pipe.unet):
        raise FloatingPointError("LoRA parameters are already non-finite; reload the pipeline cell before continuing.")

    for _ in range(cfg.dpok_epochs):
        permutation = indices[torch.randperm(batch_size)]
        for start in range(0, batch_size, cfg.minibatch_size):
            mb_idx = permutation[start:start + cfg.minibatch_size]
            mb_prompts = [prompts[i] for i in mb_idx.tolist()]
            prompt_embeds = encode_prompts(pipe, mb_prompts, cfg.negative_prompt, device, dtype)
            reference_prompt_embeds = encode_prompts(reference_pipe, mb_prompts, cfg.negative_prompt, device, dtype)
            weights = policy_weights[mb_idx].to(device=device, dtype=torch.float32)
            parameter_snapshot = clone_trainable_parameters(pipe.unet)

            optimizer.zero_grad(set_to_none=True)
            try:
                minibatch_loss = torch.zeros((), device=device, dtype=torch.float32)
                minibatch_policy_loss = torch.zeros((), device=device, dtype=torch.float32)
                minibatch_kl_loss = torch.zeros((), device=device, dtype=torch.float32)

                for step_idx, timestep in enumerate(timesteps):
                    t = torch.tensor(timestep, device=device, dtype=torch.long)
                    state = states[mb_idx, step_idx].to(device=device, dtype=dtype)
                    action = actions[mb_idx, step_idx].to(device=device, dtype=dtype)

                    noise_pred = predict_noise(pipe, state, t, prompt_embeds, cfg.guidance_scale)
                    _, log_prob = ddpm_step_with_log_prob(pipe.scheduler, noise_pred, timestep, state, prev_sample=action, eta=cfg.eta)
                    current_mean, current_std = ddpm_mean_std(pipe.scheduler, noise_pred, timestep, state, eta=cfg.eta)

                    with torch.no_grad():
                        reference_noise_pred = predict_noise(reference_pipe, state, t, reference_prompt_embeds, cfg.guidance_scale)
                        reference_mean, reference_std = ddpm_mean_std(reference_pipe.scheduler, reference_noise_pred, timestep, state, eta=cfg.eta)

                    step_kl = gaussian_kl(current_mean, current_std, reference_mean, reference_std)
                    step_policy_loss = -(cfg.reward_weight * weights * log_prob.float()).mean() / trajectory_len
                    step_kl_loss = cfg.kl_weight * step_kl.mean() / trajectory_len
                    step_loss = step_policy_loss + step_kl_loss
                    if not torch.isfinite(step_loss):
                        raise FloatingPointError("Non-finite DPOK loss; skipped this minibatch update.")
                    step_loss.backward()

                    minibatch_loss = minibatch_loss + step_loss.detach()
                    minibatch_policy_loss = minibatch_policy_loss + step_policy_loss.detach()
                    minibatch_kl_loss = minibatch_kl_loss + step_kl_loss.detach()
                    kls.append(float(step_kl.mean().detach().cpu()))

                    del state, action, noise_pred, log_prob, current_mean, current_std, reference_noise_pred, reference_mean, reference_std, step_kl, step_policy_loss, step_kl_loss, step_loss

                grad_norm = torch.nn.utils.clip_grad_norm_(trainable_parameters(pipe.unet), cfg.max_grad_norm)
                if not torch.isfinite(grad_norm):
                    raise FloatingPointError("Non-finite LoRA gradients; skipped this minibatch update.")

                optimizer.step()
                if not finite_trainable_parameters(pipe.unet):
                    restore_trainable_parameters(pipe.unet, parameter_snapshot)
                    clear_optimizer_state(optimizer)
                    raise FloatingPointError("AdamW produced non-finite LoRA parameters; restored previous weights and cleared optimizer state.")

                losses.append(float(minibatch_loss.cpu()))
                policy_losses.append(float(minibatch_policy_loss.cpu()))
                kl_losses.append(float(minibatch_kl_loss.cpu()))

            except FloatingPointError as exc:
                restore_trainable_parameters(pipe.unet, parameter_snapshot)
                optimizer.zero_grad(set_to_none=True)
                skipped_updates += 1
                print(f"Skipped DPOK minibatch: {exc}")

            if device.type == "cuda":
                torch.cuda.empty_cache()

    return {
        "loss": safe_metric_mean(losses),
        "policy_loss": safe_metric_mean(policy_losses),
        "kl_loss": safe_metric_mean(kl_losses),
        "transition_kl": safe_metric_mean(kls),
        "reward_mean": float(terminal_rewards.mean()),
        "reward_std": float(terminal_rewards.std(unbiased=False)) if terminal_rewards.numel() > 1 else 0.0,
        "skipped_updates": skipped_updates,
    }


In [ ]:
history = []
for epoch in trange(1, cfg.train_epochs + 1):
    rollout = collect_rollouts(pipe, cfg.rollouts_per_epoch)
    metrics = dpok_update(pipe, reference_pipe, rollout, optimizer)
    metrics["epoch"] = epoch
    history.append(metrics)

    if epoch % cfg.log_every == 0:
        print(metrics)

    if epoch % cfg.save_every == 0:
        ckpt_dir = Path(cfg.output_dir) / f"lora_epoch_{epoch:04d}"
        save_lora_weights(pipe, ckpt_dir)
        for i, image in enumerate(rollout["images"]):
            image.save(Path(cfg.output_dir) / f"epoch_{epoch:04d}_sample_{i:02d}.png")

    del rollout
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()


In [ ]:
rollout = collect_rollouts(pipe, cfg.rollouts_per_epoch)


In [ ]:
@torch.no_grad()
def generate_eval_images(pipe, prompts: list[str], seed: int = 123) -> list[Image.Image]:
    pipe.unet.eval()
    eval_generator = torch.Generator(device=device).manual_seed(seed)
    images = pipe(
        prompts,
        negative_prompt=[cfg.negative_prompt] * len(prompts),
        height=cfg.height,
        width=cfg.width,
        num_inference_steps=cfg.num_inference_steps,
        guidance_scale=cfg.guidance_scale,
        generator=eval_generator,
    ).images
    pipe.unet.train()
    return images


eval_prompts = [cfg.prompt, *cfg.train_prompts[:3]]
eval_images = generate_eval_images(pipe, eval_prompts)
for i, image in enumerate(eval_images):
    image.save(Path(cfg.output_dir) / f"eval_{i:02d}.png")

eval_images[0], eval_images[1], eval_images[2], eval_images[3]


In [ ]:
import matplotlib.pyplot as plt

images = eval_images
titles = eval_prompts

fig, axes = plt.subplots(2, 2, figsize=(8, 8))

for ax, img, title in zip(axes.ravel(), images, titles):
    ax.imshow(img)
    ax.set_title(title, size=10)
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
final_dir = Path(cfg.output_dir) / "lora_final"
save_lora_weights(pipe, final_dir)
print(f"Saved fine-tuned LoRA weights to {final_dir}")
print(asdict(cfg))
